# NB1 - The Evaluation Interface (V) and the Frozen-Brain Baseline

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

Recall the agent-harness framework from the intro:

> **H = (E, T, C, S, L, V)** = Execution loop, Tool registry, Context manager,
> State store, Lifecycle hooks, e**V**aluation interface.

We never touch the model weights (the "brain"). We evolve the **harness** around
it. But before anything can *evolve*, we need a **reward signal** - component
**V**. That is the whole job of this notebook.

> **Thesis of the day:** *Reflection is the gradient, the skill document is the
> parameter vector, and your eval set is the loss.* No loss -> no learning. So we
> start with the loss.

In this notebook we:
1. Meet the **environment** (a text-to-SQL task over a toy database).
2. Build the **reward signal** `score_sql` (execution match - objective, automatic).
3. Run a **frozen-brain baseline** agent and measure it.
4. Do error analysis - the failures are the raw material every later notebook learns from.

In [1]:
# Setup. We run from the notebooks/ folder, so add the repo root to the path.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_baseline_agent,
)

DB = build_db()          # deterministic rebuild; same data on every machine
print("Database ready at:", DB)

Database ready at: c:\Users\Bhaskarjit\Documents\Projects\RL-Agents-Workshop\data\shop.db


## 1. The environment: a toy "shop" database

The agent's job: translate a natural-language question into a **SQLite SELECT**
that returns the right rows. The schema is small enough to fit in a prompt and
rich enough to need joins, group-bys, subqueries, and date handling.

In [2]:
print(SCHEMA_TEXT)
print("Sample customers:", run_sql("SELECT * FROM customers LIMIT 3")[0])
print("Sample orders:   ", run_sql("SELECT * FROM orders LIMIT 3")[0])
print("Sample products: ", run_sql("SELECT * FROM products LIMIT 3")[0])

Tables:
  customers(customer_id, name, city, segment, signup_date)
      segment in (Consumer, SMB, Enterprise)
  products(product_id, name, category, price)
      category in (Electronics, Books, Clothing, Home, Toys)
  orders(order_id, customer_id, order_date, status)
      status in (completed, pending, cancelled, returned)
  order_items(item_id, order_id, product_id, quantity)

Relationships:
  orders.customer_id    -> customers.customer_id
  order_items.order_id  -> orders.order_id
  order_items.product_id-> products.product_id

Notes:
  - Dates are TEXT in 'YYYY-MM-DD' format; compare lexicographically or with strftime.
  - "Revenue" of an order item = order_items.quantity * products.price.
  - A "sale" / completed revenue counts only orders with status = 'completed'.

Sample customers: [(1, 'Aarav', 'Hyderabad', 'Consumer', '2024-01-24'), (2, 'Diya', 'Bengaluru', 'Consumer', '2024-04-05'), (3, 'Vivaan', 'Hyderabad', 'Consumer', '2024-11-24')]
Sample orders:    [(1, 8, '2025-02-1

## 2. The eval set (NL -> gold SQL)

40 tasks, labelled `easy | medium | hard`, split into **train** (we may optimize
on these) and **test** (held out - the number we actually trust).

The train/test split is the single most important discipline in the workshop:
**a self-evolving agent must improve on train without ever peeking at test.**

In [3]:
from collections import Counter
tasks = load_tasks()
print("total tasks:", len(tasks))
print("split:", dict(Counter(t["split"] for t in tasks)))
print("level:", dict(Counter(t["level"] for t in tasks)))
print()
for t in tasks[:3] + tasks[20:22]:
    print(f"#{t['id']:>2} [{t['split']}/{t['level']}] {t['question']}")
    print(f"     gold: {t['gold']}")

total tasks: 40
split: {'train': 24, 'test': 16}
level: {'easy': 10, 'medium': 10, 'hard': 20}

# 1 [train/easy] List the names of all customers in Mumbai.
     gold: SELECT name FROM customers WHERE city='Mumbai';
# 2 [train/easy] How many customers are there in total?
     gold: SELECT COUNT(*) FROM customers;
# 3 [test/easy] Show all product names in the Electronics category.
     gold: SELECT name FROM products WHERE category='Electronics';
#21 [train/hard] What is the total completed revenue, where revenue of a line item is quantity times product price and only orders with status 'completed' count?
     gold: SELECT SUM(oi.quantity*p.price) FROM order_items oi JOIN orders o ON o.order_id=oi.order_id JOIN products p ON p.product_id=oi.product_id WHERE o.status='completed';
#22 [test/hard] Which product generated the most completed revenue? Return only the product name.
     gold: SELECT p.name FROM order_items oi JOIN orders o ON o.order_id=oi.order_id JOIN products p ON p.product_

## 3. The reward signal V: `score_sql` (execution match)

We do **not** compare SQL strings - there are many correct ways to write the same
query. Instead we *execute* both the predicted and the gold query and compare the
**result sets**. Order matters only when the gold query uses `ORDER BY`.

This is exactly the "decomposed verifiable reward" idea from the ASG-SI paper:
an objective, replayable check, not an LLM's opinion.

In [4]:
t = next(x for x in tasks if x["id"] == 1)
print("Q:", t["question"])
print("gold vs gold  ->", score_sql(t["gold"], t["gold"]))                 # True
print("wrong query   ->", score_sql("SELECT city FROM customers", t["gold"]))  # False
print("syntax error  ->", score_sql("SELECT nope FROM nope", t["gold"]))      # False

Q: List the names of all customers in Mumbai.
gold vs gold  -> True
wrong query   -> False
syntax error  -> False


## 4. The frozen-brain baseline

The simplest possible agent: hand the schema + question to the LLM, zero-shot,
and parse out the SQL. **No memory, no examples, no tools, no reflection.** This
is the harness in its most bare-bones form - the number every later notebook
must beat *without touching the weights*.

In [5]:
def baseline_agent(question):
    messages = baseline_prompt(question)   # zero-shot: schema + question only
    raw = llm(messages)
    return extract_sql(raw)

# Look at the exact prompt we send (the entire "harness" right now):
print(baseline_prompt("How many customers are there in total?")[1]["content"])

Database schema:
Tables:
  customers(customer_id, name, city, segment, signup_date)
      segment in (Consumer, SMB, Enterprise)
  products(product_id, name, category, price)
      category in (Electronics, Books, Clothing, Home, Toys)
  orders(order_id, customer_id, order_date, status)
      status in (completed, pending, cancelled, returned)
  order_items(item_id, order_id, product_id, quantity)

Relationships:
  orders.customer_id    -> customers.customer_id
  order_items.order_id  -> orders.order_id
  order_items.product_id-> products.product_id

Notes:
  - Dates are TEXT in 'YYYY-MM-DD' format; compare lexicographically or with strftime.
  - "Revenue" of an order item = order_items.quantity * products.price.
  - A "sale" / completed revenue counts only orders with status = 'completed'.


Question: How many customers are there in total?
SQL:


### Run the baseline on the held-out test split

This makes real API calls with **your** key. 16 test tasks ~= 16 calls; a few
cents on `gpt-4o-mini`. The cost meter prints the spend.

In [6]:
METER.reset()
baseline = evaluate(baseline_agent, split="test", verbose=True)
print()
print("TEST accuracy:", round(baseline["accuracy"], 3))
print("by level:    ", {k: round(v["acc"], 2) for k, v in baseline["by_level"].items()})
print(METER)

  OK  [easy  ] # 3  Show all product names in the Electronics category.
  OK  [easy  ] # 5  List all distinct cities where customers live.
  OK  [easy  ] # 8  List the names of customers in the Enterprise segment.
  OK  [easy  ] # 9  How many products are in the Books category?
  OK  [medium] #12  What is the average product price per category? Return categ
  XX  [medium] #14  How many orders were placed in each month of 2025? Return th
  OK  [medium] #16  List the names of products that have never been ordered.
  XX  [medium] #18  What is the total quantity ordered for each product? Return 
  OK  [hard  ] #22  Which product generated the most completed revenue? Return o
  OK  [hard  ] #24  Which customer has the highest total completed revenue? Retu
  OK  [hard  ] #27  What is the average order value (average total revenue per o
  XX  [hard  ] #30  What is the most popular product category by total quantity 
  OK  [hard  ] #33  Which products have an average ordered quantity greater t

## 5. Error analysis - the fuel for self-evolution

Every failure below is a learning signal. In NB2 (Reflexion) the agent will
*reflect* on these to repair itself; in NB3-NB4 we distill them into reusable,
validated **skills**. Look for patterns: wrong joins? forgetting
`status='completed'`? missing `GROUP BY`? Those patterns become skills.

In [7]:
fails = [r for r in baseline["records"] if not r["correct"]]
print(f"{len(fails)} failures on test\n")
for r in fails:
    print(f"# {r['id']} [{r['level']}] {r['question']}")
    print("  pred:", r["pred"])
    print("  gold:", r["gold"])
    print()

4 failures on test

# 14 [medium] How many orders were placed in each month of 2025? Return the month and the count.
  pred: SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS order_count
FROM orders
WHERE order_date LIKE '2025-%' AND status = 'completed'
GROUP BY month
ORDER BY month;
  gold: SELECT strftime('%m', order_date) AS m, COUNT(*) FROM orders GROUP BY m;

# 18 [medium] What is the total quantity ordered for each product? Return the product name and total quantity.
  pred: SELECT products.name, SUM(order_items.quantity) AS total_quantity
FROM order_items
JOIN products ON order_items.product_id = products.product_id
JOIN orders ON order_items.order_id = orders.order_id
WHERE orders.status = 'completed'
GROUP BY products.name;
  gold: SELECT p.name, SUM(oi.quantity) FROM products p JOIN order_items oi ON oi.product_id=p.product_id GROUP BY p.product_id, p.name;

# 30 [hard] What is the most popular product category by total quantity ordered across all orders? Return onl

In [8]:
# Save the baseline number so later notebooks can show the lift over it.
import json
os.makedirs("../data", exist_ok=True)
with open("../data/baseline_test.json", "w", encoding="utf-8") as f:
    json.dump(
        {"accuracy": baseline["accuracy"],
         "by_level": {k: v["acc"] for k, v in baseline["by_level"].items()}},
        f, indent=2,
    )
print("saved baseline_test.json")

saved baseline_test.json


## Takeaways

- The **eval interface (V)** is the foundation of self-evolution. Without an
  objective reward, "self-improvement" is just vibes.
- Execution-match is a clean, replayable reward - no GPU, no LLM judge.
- The **frozen-brain baseline** is our reference. Everything from here on raises
  the test number by changing the *harness*, never the weights.

### Exercise
1. Re-run the baseline with `temperature=0.7` (edit `llm(... )`). Does accuracy
   change? What does that tell you about prompt vs. sampling?
2. Add one new hard question + gold SQL to `workshop_utils/tasks.py` and re-run.

**Next - NB2:** give the agent a memory and let it learn from these failures.
*Reflection is the gradient.*